In [ ]:

import sys
import pathlib

sys.path.append(str(pathlib.Path('..').resolve()))

%load_ext autoreload
%autoreload 2

from evaluation.utils import get_summary_df, load_model
from dataset.dataset import SetKnowledgeTrendingSinusoidsDistShift
from dataset.utils import get_dataloader

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Plot style
sns.set_style('whitegrid')
sns.set_palette('Dark2')
plt.rcParams['text.latex.preamble'] = (
    "\usepackage{lmodern} \usepackage{times} \usepackage{amssymb}"
)


In [ ]:

# Load the models
save_dirs = {
    "NP": "../saves/INPs_sinusoids/np_dist_shift_0",
    "INP": "../saves/INPs_sinusoids/inp_b_dist_shift_0",
}

model_dict = {}
config_dict = {}
for model_name, save_dir in save_dirs.items():
    model, cfg = load_model(save_dir, load_it="best")
    model.eval()
    model_dict[model_name] = model
    config_dict[model_name] = cfg

model_names = list(model_dict.keys())


In [ ]:

from argparse import Namespace

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
config = Namespace(
    min_num_context=0,
    max_num_context=100,
    num_targets=100,
    noise=0.2,
    batch_size=25,
    x_sampler='uniform',
    test_num_z_samples=32,
    dataset='set-trending-sinusoids-dist-shift',
    device=device,
)

train_dataset = SetKnowledgeTrendingSinusoidsDistShift(
    root='../data/trending-sinusoids-dist-shift', split='train', knowledge_type='b'
)
train_data_loader = get_dataloader(train_dataset, config)

test_dataset = SetKnowledgeTrendingSinusoidsDistShift(
    root='../data/trending-sinusoids-dist-shift', split='test', knowledge_type='b'
)
test_data_loader = get_dataloader(test_dataset, config)


In [ ]:

# Evaluate baseline, shuffled knowledge, and FK variants
eval_type_ls = ["raw", "informed", "shuffled", "random"]

train_summary_df, train_losses, train_output_dict = get_summary_df(
    model_dict, config_dict, train_data_loader, eval_type_ls, model_names
)
test_summary_df, test_losses, test_output_dict = get_summary_df(
    model_dict, config_dict, test_data_loader, eval_type_ls, model_names
)

train_summary_df["split"] = "train"
test_summary_df["split"] = "test"


In [ ]:

import re

def select_best_per_context(df, model_name, base_eval, kind, metric_col="mean"):
    """
    Pick best hyperparameter (min NLL) per num_context for a given model + base eval.
    kind: "fk_lambda" or "fkdr_alpha".
    """
    if kind == "fk_lambda":
        pat = rf"^{re.escape(base_eval)}_fk_lambda_([0-9\.]+)$"
    else:
        pat = rf"^{re.escape(base_eval)}_fkdr_alpha_([0-9\.]+)$"

    sub = df[(df.model_name == model_name) & (df.eval_type.str.match(pat, na=False))].copy()
    if sub.empty:
        return sub

    sub["hyper"] = sub["eval_type"].str.extract(pat).astype(float)
    best = (
        sub.sort_values(["num_context", metric_col], ascending=[True, True])
        .groupby("num_context", as_index=False)
        .first()
    )
    return best


In [ ]:

# Baseline NP vs INP on train and test (average log-likelihood)
combined_df = pd.concat([train_summary_df, test_summary_df], ignore_index=True, sort=False)
base_map = {
    "raw": r"NP: $\mathcal{K} = \varnothing$",
    "informed": r"INP: $\mathcal{K} \neq \varnothing$",
}

base_df = combined_df[
    (combined_df.eval_type.isin(base_map)) & (combined_df.model_name.isin(["NP", "INP"]))
].copy()
base_df["eval_label"] = base_df["eval_type"].map(base_map)
base_df["avg_log_like"] = -base_df["mean"]

fig, ax = plt.subplots(figsize=(4.5, 3.4))
sns.lineplot(
    data=base_df,
    x="num_context",
    y="avg_log_like",
    hue="eval_label",
    style="split",
    style_order=["train", "test"],
    markers=True,
    ax=ax,
)
ax.set_ylabel("Average log-likelihood")
ax.set_xlabel("Number of context datapoints")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="")
plt.tight_layout()
plt.show()


In [ ]:

# Test split: NP vs INP vs FK best
df = test_summary_df.copy().dropna(subset=["mean"])
df["avg_log_like"] = -df["mean"]

# Baselines
baseline_rows = []
np_base = df[(df.model_name == "NP") & (df.eval_type == "raw")].copy()
np_base["eval_label"] = r"NP: $\mathcal{K} = \varnothing$"
baseline_rows.append(np_base)

inp_base = df[(df.model_name == "INP") & (df.eval_type == "informed")].copy()
inp_base["eval_label"] = r"INP: $\mathcal{K} \neq \varnothing$"
baseline_rows.append(inp_base)

inp_shuf = df[(df.model_name == "INP") & (df.eval_type == "shuffled")].copy()
if not inp_shuf.empty:
    inp_shuf["eval_label"] = "INP: K shuffled"
    baseline_rows.append(inp_shuf)

inp_rand = df[(df.model_name == "INP") & (df.eval_type == "random")].copy()
if not inp_rand.empty:
    inp_rand["eval_label"] = "INP: K random"
    baseline_rows.append(inp_rand)

base_plot = pd.concat(baseline_rows, ignore_index=True)

# FK best selections
np_fk_best = select_best_per_context(df, model_name="NP", base_eval="raw", kind="fk_lambda")
if not np_fk_best.empty:
    np_fk_best["avg_log_like"] = -np_fk_best["mean"]
    np_fk_best["eval_label"] = np_fk_best["hyper"].apply(lambda v: f"NP + FK (best λ={v:g})")

inp_fkdr_best = select_best_per_context(df, model_name="INP", base_eval="informed", kind="fkdr_alpha")
if not inp_fkdr_best.empty:
    inp_fkdr_best["avg_log_like"] = -inp_fkdr_best["mean"]
    inp_fkdr_best["eval_label"] = inp_fkdr_best["hyper"].apply(lambda v: f"INP + FK-DR (best α={v:g})")

inp_shuf_fk_best = select_best_per_context(df, model_name="INP", base_eval="shuffled", kind="fk_lambda")
if not inp_shuf_fk_best.empty:
    inp_shuf_fk_best["avg_log_like"] = -inp_shuf_fk_best["mean"]
    inp_shuf_fk_best["eval_label"] = inp_shuf_fk_best["hyper"].apply(lambda v: f"INP shuffled + FK (best λ={v:g})")

plot_parts = [base_plot]
for extra in [np_fk_best, inp_fkdr_best, inp_shuf_fk_best]:
    if isinstance(extra, pd.DataFrame) and not extra.empty:
        plot_parts.append(extra)
plot_df = pd.concat(plot_parts, ignore_index=True, sort=False)

plt.figure(figsize=(5.0, 3.6))
sns.lineplot(
    data=plot_df,
    x="num_context",
    y="avg_log_like",
    hue="eval_label",
    marker="o",
)
plt.ylabel("Average log-likelihood (test)")
plt.xlabel("Number of context datapoints")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="")
plt.tight_layout()
plt.show()

# Delta vs NP baseline
np_curve = np_base[["num_context", "avg_log_like"]].rename(columns={"avg_log_like": "avg_log_like_np"})
curves = []
if not np_fk_best.empty:
    c = np_fk_best[["num_context", "avg_log_like"]].copy()
    c["label"] = "NP + FK (best)"
    curves.append(c)

inp_curve = inp_base[["num_context", "avg_log_like"]].copy()
inp_curve["label"] = "INP informed"
curves.append(inp_curve)

if not inp_shuf.empty:
    c = inp_shuf[["num_context", "avg_log_like"]].copy()
    c["label"] = "INP shuffled"
    curves.append(c)

if not inp_rand.empty:
    c = inp_rand[["num_context", "avg_log_like"]].copy()
    c["label"] = "INP random"
    curves.append(c)

if not inp_fkdr_best.empty:
    c = inp_fkdr_best[["num_context", "avg_log_like"]].copy()
    c["label"] = "INP + FK-DR (best)"
    curves.append(c)

if not inp_shuf_fk_best.empty:
    c = inp_shuf_fk_best[["num_context", "avg_log_like"]].copy()
    c["label"] = "INP shuffled + FK (best)"
    curves.append(c)

delta_rows = []
for c in curves:
    m = c.merge(np_curve, on="num_context")
    m["delta_vs_np"] = m["avg_log_like"] - m["avg_log_like_np"]
    m = m[["num_context", "label", "delta_vs_np"]]
    delta_rows.append(m)

delta_df = pd.concat(delta_rows, ignore_index=True)

plt.figure(figsize=(4.8, 3.4))
sns.lineplot(
    data=delta_df,
    x="num_context",
    y="delta_vs_np",
    hue="label",
    marker="o",
)
plt.axhline(0.0, linewidth=1, color="black", linestyle="--")
plt.ylabel("Δ avg log-likelihood vs NP baseline (test)")
plt.xlabel("Number of context datapoints")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, title="")
plt.tight_layout()
plt.show()


In [ ]:

# Summaries of best FK hyperparameters per context (test split)
best_tables = []
for kind, label in [("fk_lambda", "FK λ"), ("fkdr_alpha", "FK-DR α")]:
    for model_name, base_eval in [("NP", "raw"), ("INP", "informed"), ("INP", "shuffled")]:
        best = select_best_per_context(test_summary_df, model_name=model_name, base_eval=base_eval, kind=kind)
        if best.empty:
            continue
        tbl = best[["num_context", "hyper"]].rename(columns={"hyper": label}).copy()
        tbl["model"] = model_name
        tbl["eval_type"] = base_eval
        best_tables.append(tbl)

if best_tables:
    best_hyper_df = pd.concat(best_tables, ignore_index=True)
    display(
        best_hyper_df.pivot_table(
            index=["model", "eval_type"],
            columns="num_context",
            values=[c for c in best_hyper_df.columns if c in ["FK λ", "FK-DR α"]],
        )
    )
else:
    print("No FK rows available to summarize.")
